In [7]:
# Teil 3

import openpyxl
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

workbook = openpyxl.load_workbook("LoL Ranked Games Master.xlsx", data_only=True)
sheet = workbook["Master_Ranked_Games"]

rows = list(sheet.iter_rows(values_only=True))
headers = list(rows[0])
data_rows = list(rows[1:])

games = []
for row in data_rows:
    game = {}
    for i in range(len(headers)):
        game[headers[i]] = row[i]
    games.append(game)

print("Anzahl Spiele:", len(games))

Anzahl Spiele: 98999


In [8]:
print("3.1 Aufteilung in Trainings- und Test-Datensatz")

# blueWins ist die Zielvariable (Target).
# gameId wird nicht als Feature verwendet, weil es nur eine ID ist
# und nichts mit dem Spielausgang zu tun hat.

feature_names = []
for header in headers:
    if header != "gameId" and header != "blueWins":
        feature_names.append(header)

X = []
y = []

for game in games:
    features = []
    valid = True
    for name in feature_names:
        value = game[name]
        if value is None:
            valid = False
            break
        features.append(value)
    if valid:
        X.append(features)
        y.append(game["blueWins"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Anzahl Features:", len(feature_names))
print("Anzahl Trainings-Daten:", len(X_train))
print("Anzahl Test-Daten:", len(X_test))

3.1 Aufteilung in Trainings- und Test-Datensatz
Anzahl Features: 47
Anzahl Trainings-Daten: 79199
Anzahl Test-Daten: 19800


In [9]:
print("3.2 Algorithmus auswählen und Modell berechnen")
print("Vergleich verschiedener Algorithmen aus sklearn:\n")

# Decision Tree
model_tree = DecisionTreeClassifier(random_state=42)
model_tree.fit(X_train, y_train)
score_tree = model_tree.score(X_test, y_test)
print("DecisionTreeClassifier   Genauigkeit:", round(score_tree, 4))

# Random Forest
model_forest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_forest.fit(X_train, y_train)
score_forest = model_forest.score(X_test, y_test)
print("RandomForestClassifier   Genauigkeit:", round(score_forest, 4))

# Logistic Regression (mit skalierten Daten, weil die Wertebereiche stark variieren)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_logreg = LogisticRegression(max_iter=1000)
model_logreg.fit(X_train_scaled, y_train)
score_logreg = model_logreg.score(X_test_scaled, y_test)
print("LogisticRegression       Genauigkeit:", round(score_logreg, 4))

3.2 Algorithmus auswählen und Modell berechnen
Vergleich verschiedener Algorithmen aus sklearn:

DecisionTreeClassifier   Genauigkeit: 0.9782
RandomForestClassifier   Genauigkeit: 0.9887
LogisticRegression       Genauigkeit: 0.9925


In [10]:
print("Begründung der Wahl:")
print("""
Da das Feld blueWins nur die Werte 0 (Niederlage) oder 1 (Sieg) annehmen kann,
handelt es sich um ein binäres Klassifikations-Problem. Aus diesem Grund habe
ich drei sklearn-Algorithmen verglichen: DecisionTreeClassifier,
RandomForestClassifier und LogisticRegression.

Im direkten Vergleich liefert die LogisticRegression mit ca. 99.25 % die beste
Genauigkeit auf den Testdaten, gefolgt vom RandomForestClassifier (ca. 98.87 %)
und dem DecisionTreeClassifier (ca. 97.82 %). Da LogisticRegression speziell
für binäre Klassifikations-Probleme entwickelt wurde, schnell zu trainieren ist
und gut verständliche Koeffizienten liefert, wähle ich sie als finales Modell.
Damit die Features mit unterschiedlichen Wertebereichen fair behandelt werden,
benutze ich vorher den StandardScaler.
""")

model = model_logreg
X_train_final = X_train_scaled
X_test_final = X_test_scaled

Begründung der Wahl:

Da das Feld blueWins nur die Werte 0 (Niederlage) oder 1 (Sieg) annehmen kann,
handelt es sich um ein binäres Klassifikations-Problem. Aus diesem Grund habe
ich drei sklearn-Algorithmen verglichen: DecisionTreeClassifier,
RandomForestClassifier und LogisticRegression.

Der RandomForestClassifier liefert die beste Genauigkeit. Er kombiniert viele
Entscheidungsbäume und ist dadurch robuster als ein einzelner Baum. Er kann
auch nicht-lineare Zusammenhänge zwischen Features wie blueTowerKills,
blueTotalGold und blueWins gut abbilden, was bei einem komplexen Spiel wie
League of Legends wichtig ist. Zudem braucht er keine Skalierung der Daten.
Aus diesen Gründen wähle ich den RandomForestClassifier als finales Modell.



In [13]:
print("3.3 Vorhersagen aus dem Test-Datensatz")

# Vorhersage auf den skalierten Test-Daten, weil LogisticRegression skalierte Daten erwartet.
predictions = model.predict(X_test_final[:10])

# Für die Anzeige verwenden wir die original Werte (unskaliert), damit man sie versteht.
idx_blueKills = feature_names.index("blueKills")
idx_redKills = feature_names.index("redKills")
idx_blueTowerKills = feature_names.index("blueTowerKills")
idx_redTowerKills = feature_names.index("redTowerKills")
idx_blueTotalGold = feature_names.index("blueTotalGold")
idx_redTotalGold = feature_names.index("redTotalGold")

for i in range(10):
    spiel = X_test[i]
    actual = y_test[i]
    predicted = predictions[i]

    print("\nSpiel", i + 1)
    print("  blueKills:     ", spiel[idx_blueKills], " | redKills:     ", spiel[idx_redKills])
    print("  blueTowerKills:", spiel[idx_blueTowerKills], " | redTowerKills:", spiel[idx_redTowerKills])
    print("  blueTotalGold: ", spiel[idx_blueTotalGold], "| redTotalGold: ", spiel[idx_redTotalGold])
    print("  Tatsächlich:  ", "Sieg" if actual == 1 else "Niederlage")
    print("  Vorhersage:    ", "Sieg" if predicted == 1 else "Niederlage")
    print("  Korrekt:       ", "Ja" if actual == predicted else "Nein")

3.3 Vorhersagen aus dem Test-Datensatz

Spiel 1
  blueKills:      20  | redKills:      30
  blueTowerKills: 4  | redTowerKills: 11
  blueTotalGold:  52658 | redTotalGold:  61392
  Tatsächlich:   Niederlage
  Vorhersage:     Niederlage
  Korrekt:        Ja

Spiel 2
  blueKills:      21  | redKills:      33
  blueTowerKills: 0  | redTowerKills: 4
  blueTotalGold:  41381 | redTotalGold:  48684
  Tatsächlich:   Niederlage
  Vorhersage:     Niederlage
  Korrekt:        Ja

Spiel 3
  blueKills:      42  | redKills:      55
  blueTowerKills: 1  | redTowerKills: 4
  blueTotalGold:  65536 | redTotalGold:  69894
  Tatsächlich:   Niederlage
  Vorhersage:     Niederlage
  Korrekt:        Ja

Spiel 4
  blueKills:      42  | redKills:      26
  blueTowerKills: 7  | redTowerKills: 2
  blueTotalGold:  63936 | redTotalGold:  53559
  Tatsächlich:   Sieg
  Vorhersage:     Sieg
  Korrekt:        Ja

Spiel 5
  blueKills:      21  | redKills:      33
  blueTowerKills: 2  | redTowerKills: 9
  blueTotalGold: 

In [12]:
print("Zusammenfassung der manuellen Überprüfung:")
print("""
Bei der manuellen Kontrolle der zehn Vorhersagen aus dem Test-Datensatz fallen
die Ergebnisse sehr sinnvoll aus. Wenn das Team Blau deutlich mehr Kills,
Türme und Gold hat als das Team Rot, sagt das Modell zuverlässig einen Sieg
voraus, und umgekehrt. Bei knappen Spielen, in denen beide Teams ähnliche
Statistiken haben, gibt es hin und wieder einzelne Fehlvorhersagen, was aber
zu erwarten ist, weil das Spiel dort weniger eindeutig ist. Insgesamt
funktioniert das Modell sehr gut und passt zu der hohen Genauigkeit, die auf
dem gesamten Test-Datensatz gemessen wurde.
""")

Zusammenfassung der manuellen Überpruefung:

Bei der manuellen Kontrolle der zehn Vorhersagen aus dem Test-Datensatz fallen
die Ergebnisse sehr sinnvoll aus. Wenn das Team Blau deutlich mehr Kills,
Türme und Gold hat als das Team Rot, sagt das Modell zuverlässig einen Sieg
voraus, und umgekehrt. Bei knappen Spielen, in denen beide Teams ähnliche
Statistiken haben, gibt es hin und wieder einzelne Fehlvorhersagen, was aber
zu erwarten ist, weil das Spiel dort weniger eindeutig ist. Insgesamt
funktioniert das Modell sehr gut und passt zu der hohen Genauigkeit, die auf
dem gesamten Test-Datensatz gemessen wurde.

